In [0]:
import pandas as pd
import numpy as np

df = spark.table("default.refined_mtc_dataset").toPandas()

print("Shape:", df.shape)
print(df["target"].value_counts())

Shape: (1107, 197)
target
0    814
1    293
Name: count, dtype: int64


In [0]:
identifier_columns = [
    "dataset_folder",
    "mtc_file",
    "segment_id",
    "start",
    "end",
    "raw_label"
]

X = df.drop(columns=identifier_columns + ["target"])
y = df["target"]
groups = df["dataset_folder"]

X = X.select_dtypes(include=np.number)

print("Features:", X.shape)
print("Experiments:", groups.nunique())

Features: (1107, 190)
Experiments: 24


In [0]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Overlap:", set(groups_train).intersection(set(groups_test)))

Train shape: (942, 190)
Test shape: (165, 190)
Overlap: set()


In [0]:
baseline_metrics_df = spark.table(
    "default.cnc_baseline_metrics"
).toPandas()

baseline_metrics = baseline_metrics_df.iloc[0].to_dict()

print("Baseline metrics:")
print(baseline_metrics)

Baseline metrics:
{'accuracy': 0.8606060606060606, 'f1_score': 0.8866995073891626, 'precision': 0.8411214953271028, 'recall': 0.9375, 'roc_auc': 0.8463164251207729}


In [0]:
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [0]:
EXPERIMENT_NAME = "/Shared/cnc_milling_anomaly_detection"

mlflow.set_experiment(EXPERIMENT_NAME)

print("Experiment:", EXPERIMENT_NAME)

Experiment: /Shared/cnc_milling_anomaly_detection


In [0]:
improved_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "classifier",
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

In [0]:
parameter_distributions = {
    "classifier__n_estimators": [
        300,
        500,
        700,
        1000
    ],

    "classifier__max_depth": [
        None,
        8,
        12,
        16,
        20
    ],

    "classifier__min_samples_split": [
        2,
        5,
        10,
        15
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        3,
        5,
        8
    ],

    "classifier__max_features": [
        "sqrt",
        "log2",
        0.5,
        0.75
    ],

    "classifier__bootstrap": [
        True,
        False
    ]
}

In [0]:
group_cv = GroupKFold(n_splits=5)

In [0]:
search = RandomizedSearchCV(
    estimator=improved_pipeline,
    param_distributions=parameter_distributions,
    n_iter=30,
    scoring="f1",
    cv=group_cv,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

In [0]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold

smaller_parameter_distributions = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [8, 12, 16, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2", 0.5],
    "classifier__bootstrap": [True, False]
}

search = RandomizedSearchCV(
    estimator=improved_pipeline,
    param_distributions=smaller_parameter_distributions,
    n_iter=8,
    scoring="f1",
    cv=GroupKFold(n_splits=4),
    random_state=42,
    n_jobs=2,
    verbose=2,
    return_train_score=False
)

In [0]:
search.fit(
    X_train,
    y_train,
    groups=groups_train
)

print("Best CV F1:", search.best_score_)
print("Best parameters:")
print(search.best_params_)

Fitting 4 folds for each of 8 candidates, totalling 32 fits


E0000 00:00:1785005962.786611    3724 thd.cc:160] pthread_create failed: Resource temporarily unavailable


In [0]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold

smaller_parameter_distributions = {
    "classifier__n_estimators": [150, 200, 300],
    "classifier__max_depth": [8, 12, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"],
    "classifier__bootstrap": [True]
}

search = RandomizedSearchCV(
    estimator=improved_pipeline,
    param_distributions=smaller_parameter_distributions,
    n_iter=5,
    scoring="f1",
    cv=GroupKFold(n_splits=3),
    random_state=42,
    n_jobs=1,
    verbose=2,
    return_train_score=False
)

In [0]:
search.fit(
    X_train,
    y_train,
    groups=groups_train
)

print("Best CV F1:", search.best_score_)
print("Best parameters:")
print(search.best_params_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END classifier__bootstrap=True, classifier__max_depth=None, classifier__max_features=log2, classifier__min_samples_leaf=4, classifier__min_samples_split=5, classifier__n_estimators=300; total time=   0.8s
[CV] END classifier__bootstrap=True, classifier__max_depth=None, classifier__max_features=log2, classifier__min_samples_leaf=4, classifier__min_samples_split=5, classifier__n_estimators=300; total time=   0.8s
[CV] END classifier__bootstrap=True, classifier__max_depth=None, classifier__max_features=log2, classifier__min_samples_leaf=4, classifier__min_samples_split=5, classifier__n_estimators=300; total time=   0.9s
[CV] END classifier__bootstrap=True, classifier__max_depth=None, classifier__max_features=sqrt, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=200; total time=   0.6s
[CV] END classifier__bootstrap=True, classifier__max_depth=None, classifier__max_features=sqrt, clas

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

improved_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "classifier",
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        )
    )
])

In [0]:
improved_model = search.best_estimator_

improved_predictions = improved_model.predict(X_test)
improved_probabilities = improved_model.predict_proba(X_test)[:, 1]

In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

improved_metrics = {
    "accuracy": accuracy_score(y_test, improved_predictions),
    "precision": precision_score(
        y_test,
        improved_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        improved_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        improved_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        improved_probabilities
    )
}

print("Improved model metrics:")
print(improved_metrics)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        improved_predictions,
        digits=4
    )
)

print("Confusion matrix:")
print(confusion_matrix(y_test, improved_predictions))

Improved model metrics:
{'accuracy': 0.7454545454545455, 'precision': 0.717741935483871, 'recall': 0.9270833333333334, 'f1_score': 0.8090909090909091, 'roc_auc': np.float64(0.9002113526570048)}

Classification report:
              precision    recall  f1-score   support

           0     0.8293    0.4928    0.6182        69
           1     0.7177    0.9271    0.8091        96

    accuracy                         0.7455       165
   macro avg     0.7735    0.7099    0.7136       165
weighted avg     0.7644    0.7455    0.7293       165

Confusion matrix:
[[34 35]
 [ 7 89]]


In [0]:
import mlflow
import mlflow.sklearn

EXPERIMENT_NAME = "/Shared/cnc_milling_anomaly_detection"
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2857151153212128', creation_time=1785004380421, experiment_id='2857151153212128', last_update_time=1785004433279, lifecycle_stage='active', name='/Shared/cnc_milling_anomaly_detection', tags={'mlflow.experiment.sourceName': '/Shared/cnc_milling_anomaly_detection',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'aswinachu304@gmail.com',
 'mlflow.ownerId': '73095880444919'}>

In [0]:
with mlflow.start_run(
    run_name="improved_random_forest_v2"
) as improved_run:

    mlflow.log_params(search.best_params_)

    mlflow.log_metric(
        "cv_best_f1",
        float(search.best_score_)
    )

    mlflow.log_metrics({
        key: float(value)
        for key, value in improved_metrics.items()
    })

    mlflow.log_param(
        "improvement_method",
        "group_aware_randomized_search"
    )

    mlflow.log_param(
        "number_of_features",
        X_train.shape[1]
    )

    mlflow.log_param(
        "train_rows",
        X_train.shape[0]
    )

    mlflow.log_param(
        "test_rows",
        X_test.shape[0]
    )

    mlflow.sklearn.log_model(
        improved_model,
        artifact_path="model",
        input_example=X_train.head(3)
    )

    improved_run_id = improved_run.info.run_id

print("Improved run ID:", improved_run_id)

2026/07/25 19:02:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-700d842f-f782.cloud.databricks.com/ml/experiments/2857151153212128/models/m-7dbe773f69ac48979f8b0fff04f96fd5?o=7474646333398352
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for 

Improved run ID: 23635898da0641379d40110ac3c1003a


In [0]:
baseline_metrics_df = spark.table(
    "default.cnc_baseline_metrics"
).toPandas()

baseline_metrics = baseline_metrics_df.iloc[0].to_dict()

print("Baseline metrics:")
print(baseline_metrics)

Baseline metrics:
{'accuracy': 0.8606060606060606, 'f1_score': 0.8866995073891626, 'precision': 0.8411214953271028, 'recall': 0.9375, 'roc_auc': 0.8463164251207729}


In [0]:
baseline_f1 = float(baseline_metrics["f1_score"])
baseline_recall = float(baseline_metrics["recall"])

improved_f1 = float(improved_metrics["f1_score"])
improved_recall = float(improved_metrics["recall"])

f1_improved = improved_f1 > baseline_f1

recall_acceptable = (
    improved_recall >= baseline_recall - 0.02
)

should_promote = (
    f1_improved
    and recall_acceptable
)

print("Baseline F1:", baseline_f1)
print("Improved F1:", improved_f1)

print("Baseline recall:", baseline_recall)
print("Improved recall:", improved_recall)

print("F1 improved:", f1_improved)
print("Recall acceptable:", recall_acceptable)
print("Promote Version 2:", should_promote)

Baseline F1: 0.8866995073891626
Improved F1: 0.8090909090909091
Baseline recall: 0.9375
Improved recall: 0.9270833333333334
F1 improved: False
Recall acceptable: True
Promote Version 2: False


In [0]:
baseline_f1 = 0.8866995073891626
baseline_recall = 0.9375

challenger_f1 = 0.84
challenger_recall = 1.0

promote = (
    challenger_f1 > baseline_f1
    and challenger_recall >= baseline_recall - 0.02
)

print("Champion model: Decision Tree")
print("Challenger model: Voting Ensemble")
print("Baseline F1:", round(baseline_f1, 4))
print("Challenger F1:", round(challenger_f1, 4))
print("Baseline recall:", round(baseline_recall, 4))
print("Challenger recall:", round(challenger_recall, 4))
print("Promote challenger:", promote)
print("Production decision: Keep Version 1 as Champion")

Champion model: Decision Tree
Challenger model: Voting Ensemble
Baseline F1: 0.8867
Challenger F1: 0.84
Baseline recall: 0.9375
Challenger recall: 1.0
Promote challenger: False
Production decision: Keep Version 1 as Champion


In [0]:
REGISTERED_MODEL_NAME = "cnc_milling_anomaly_model"

improved_model_uri = (
    f"runs:/{improved_run_id}/model"
)

new_model_version = mlflow.register_model(
    model_uri=improved_model_uri,
    name=REGISTERED_MODEL_NAME
)

print("Registered version:", new_model_version.version)

Registered model 'cnc_milling_anomaly_model' already exists. Creating a new version of this model...
2026/07/25 19:03:15 WARNING mlflow.tracking._model_registry.fluent: Run with id 23635898da0641379d40110ac3c1003a has no artifacts at artifact path 'model', registering model based on models:/m-7dbe773f69ac48979f8b0fff04f96fd5 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered version: 2


🔗 Created version '2' of model 'workspace.default.cnc_milling_anomaly_model': https://dbc-700d842f-f782.cloud.databricks.com/explore/data/models/workspace/default/cnc_milling_anomaly_model/version/2?o=7474646333398352


In [0]:
from mlflow import MlflowClient

client = MlflowClient()

if should_promote:
    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias="Champion",
        version=new_model_version.version
    )

    print(
        f"Version {new_model_version.version} "
        "promoted to Champion."
    )
else:
    print(
        f"Version {new_model_version.version} "
        "registered but not promoted."
    )

Version 2 registered but not promoted.


In [0]:
champion = client.get_model_version_by_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Champion"
)

print("Current Champion version:", champion.version)
print("Champion run ID:", champion.run_id)

Current Champion version: 1
Champion run ID: b5ee6033feee49b6a3355903b5256cc9


In [0]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold

improved_tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

tree_parameters = {
    "classifier__criterion": ["gini", "entropy", "log_loss"],
    "classifier__max_depth": [4, 5, 6, 7, 8, 10, 12, None],
    "classifier__min_samples_split": [2, 5, 10, 15, 20],
    "classifier__min_samples_leaf": [1, 2, 3, 5, 8, 10],
    "classifier__max_features": [None, "sqrt", "log2"],
    "classifier__ccp_alpha": [0.0, 0.0001, 0.001, 0.005, 0.01]
}

tree_search = RandomizedSearchCV(
    estimator=improved_tree_pipeline,
    param_distributions=tree_parameters,
    n_iter=20,
    scoring="f1",
    cv=GroupKFold(n_splits=4),
    random_state=42,
    n_jobs=1,
    verbose=2
)

tree_search.fit(
    X_train,
    y_train,
    groups=groups_train
)

print("Best CV F1:", tree_search.best_score_)
print("Best parameters:")
print(tree_search.best_params_)

Fitting 4 folds for each of 20 candidates, totalling 80 fits
[CV] END classifier__ccp_alpha=0.005, classifier__criterion=entropy, classifier__max_depth=4, classifier__max_features=log2, classifier__min_samples_leaf=3, classifier__min_samples_split=2; total time=   0.1s
[CV] END classifier__ccp_alpha=0.005, classifier__criterion=entropy, classifier__max_depth=4, classifier__max_features=log2, classifier__min_samples_leaf=3, classifier__min_samples_split=2; total time=   0.0s
[CV] END classifier__ccp_alpha=0.005, classifier__criterion=entropy, classifier__max_depth=4, classifier__max_features=log2, classifier__min_samples_leaf=3, classifier__min_samples_split=2; total time=   0.0s
[CV] END classifier__ccp_alpha=0.005, classifier__criterion=entropy, classifier__max_depth=4, classifier__max_features=log2, classifier__min_samples_leaf=3, classifier__min_samples_split=2; total time=   0.0s
[CV] END classifier__ccp_alpha=0.0, classifier__criterion=entropy, classifier__max_depth=5, classifier_

In [0]:
from sklearn.model_selection import cross_val_predict
import numpy as np
from sklearn.metrics import f1_score, recall_score

best_tree = tree_search.best_estimator_

training_probabilities = cross_val_predict(
    best_tree,
    X_train,
    y_train,
    groups=groups_train,
    cv=GroupKFold(n_splits=4),
    method="predict_proba",
    n_jobs=1
)[:, 1]

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.02):
    predictions = (
        training_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": float(threshold),
        "f1_score": f1_score(
            y_train,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_train,
            predictions,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

eligible_thresholds = threshold_df[
    threshold_df["recall"] >= 0.90
]

if not eligible_thresholds.empty:
    best_threshold_row = eligible_thresholds.sort_values(
        by=["f1_score", "recall"],
        ascending=False
    ).iloc[0]
else:
    best_threshold_row = threshold_df.sort_values(
        by=["f1_score", "recall"],
        ascending=False
    ).iloc[0]

best_threshold = float(
    best_threshold_row["threshold"]
)

print("Selected threshold:", best_threshold)
display(
    threshold_df.sort_values(
        "f1_score",
        ascending=False
    ).head(10)
)

Selected threshold: 0.47999999999999987


threshold,f1_score,recall
0.47999999999999987,0.6925064599483204,0.6802030456852792
0.5999999999999999,0.6925064599483204,0.6802030456852792
0.5799999999999998,0.6925064599483204,0.6802030456852792
0.5599999999999998,0.6925064599483204,0.6802030456852792
0.5399999999999998,0.6925064599483204,0.6802030456852792
0.5199999999999998,0.6925064599483204,0.6802030456852792
0.49999999999999983,0.6925064599483204,0.6802030456852792
0.4399999999999999,0.6854460093896714,0.7411167512690355
0.41999999999999993,0.6854460093896714,0.7411167512690355
0.29999999999999993,0.6854460093896714,0.7411167512690355


In [0]:
import pandas as pd

challenger_summary = pd.DataFrame([
    {
        "Model": "Champion Decision Tree",
        "F1 Score": 0.8867,
        "Recall": 0.9375,
        "Decision": "Champion"
    },
    {
        "Model": "Extra Trees",
        "F1 Score": 0.8087,
        "Recall": 0.9688,
        "Decision": "Rejected"
    },
    {
        "Model": "XGBoost",
        "F1 Score": 0.8190,
        "Recall": 0.9896,
        "Decision": "Rejected"
    },
    {
        "Model": "Voting Ensemble",
        "F1 Score": 0.8400,
        "Recall": 1.0000,
        "Decision": "Rejected"
    }
])

print(
    challenger_summary
    .to_string(index=False)
)

                 Model  F1 Score  Recall Decision
Champion Decision Tree    0.8867  0.9375 Champion
           Extra Trees    0.8087  0.9688 Rejected
               XGBoost    0.8190  0.9896 Rejected
       Voting Ensemble    0.8400  1.0000 Rejected


In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

best_tree.fit(X_train, y_train)

version3_probabilities = best_tree.predict_proba(
    X_test
)[:, 1]

version3_predictions = (
    version3_probabilities >= best_threshold
).astype(int)

version3_metrics = {
    "accuracy": accuracy_score(
        y_test,
        version3_predictions
    ),
    "precision": precision_score(
        y_test,
        version3_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        version3_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        version3_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        version3_probabilities
    )
}

print("Version 3 metrics:")
print(version3_metrics)

print("\nClassification report:")
print(classification_report(
    y_test,
    version3_predictions,
    digits=4
))

print("Confusion matrix:")
print(confusion_matrix(
    y_test,
    version3_predictions
))

Version 3 metrics:
{'accuracy': 0.509090909090909, 'precision': 0.5663716814159292, 'recall': 0.6666666666666666, 'f1_score': 0.6124401913875598, 'roc_auc': np.float64(0.3883605072463768)}

Classification report:
              precision    recall  f1-score   support

           0     0.3846    0.2899    0.3306        69
           1     0.5664    0.6667    0.6124        96

    accuracy                         0.5091       165
   macro avg     0.4755    0.4783    0.4715       165
weighted avg     0.4904    0.5091    0.4946       165

Confusion matrix:
[[20 49]
 [32 64]]


# Version 4 – Tuned Gradient Boosting

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold

gb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", GradientBoostingClassifier(
        random_state=42
    ))
])

gb_parameters = {
    "classifier__n_estimators": [100, 150, 200],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__max_depth": [1, 2, 3],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__subsample": [0.8, 1.0],
    "classifier__max_features": [None, "sqrt"]
}

gb_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_parameters,
    n_iter=8,
    scoring="f1",
    cv=GroupKFold(n_splits=3),
    random_state=42,
    n_jobs=1,
    verbose=2
)

gb_search.fit(
    X_train,
    y_train,
    groups=groups_train
)

print("Best CV F1:", gb_search.best_score_)
print("Best parameters:")
print(gb_search.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END classifier__learning_rate=0.03, classifier__max_depth=1, classifier__max_features=sqrt, classifier__min_samples_leaf=4, classifier__min_samples_split=10, classifier__n_estimators=100, classifier__subsample=0.8; total time=   0.2s
[CV] END classifier__learning_rate=0.03, classifier__max_depth=1, classifier__max_features=sqrt, classifier__min_samples_leaf=4, classifier__min_samples_split=10, classifier__n_estimators=100, classifier__subsample=0.8; total time=   0.2s
[CV] END classifier__learning_rate=0.03, classifier__max_depth=1, classifier__max_features=sqrt, classifier__min_samples_leaf=4, classifier__min_samples_split=10, classifier__n_estimators=100, classifier__subsample=0.8; total time=   0.2s
[CV] END classifier__learning_rate=0.05, classifier__max_depth=2, classifier__max_features=None, classifier__min_samples_leaf=1, classifier__min_samples_split=2, classifier__n_estimators=150, classifier__subsample=1.0; tota

In [0]:
gb_model = gb_search.best_estimator_

gb_predictions = gb_model.predict(X_test)
gb_probabilities = gb_model.predict_proba(X_test)[:, 1]

gb_metrics = {
    "accuracy": accuracy_score(y_test, gb_predictions),
    "precision": precision_score(
        y_test,
        gb_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        gb_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        gb_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        gb_probabilities
    )
}

print("Version 4 metrics:")
print(gb_metrics)

print("\nClassification report:")
print(classification_report(
    y_test,
    gb_predictions,
    digits=4
))

print("Confusion matrix:")
print(confusion_matrix(
    y_test,
    gb_predictions
))

Version 4 metrics:
{'accuracy': 0.7636363636363637, 'precision': 0.7567567567567568, 'recall': 0.875, 'f1_score': 0.8115942028985508, 'roc_auc': np.float64(0.8842089371980677)}

Classification report:
              precision    recall  f1-score   support

           0     0.7778    0.6087    0.6829        69
           1     0.7568    0.8750    0.8116        96

    accuracy                         0.7636       165
   macro avg     0.7673    0.7418    0.7473       165
weighted avg     0.7655    0.7636    0.7578       165

Confusion matrix:
[[42 27]
 [12 84]]


In [0]:
version4_should_promote = (
    gb_metrics["f1_score"] > baseline_f1
    and gb_metrics["recall"] >= baseline_recall - 0.02
)

print("Baseline F1:", baseline_f1)
print("Version 4 F1:", gb_metrics["f1_score"])

print("Baseline recall:", baseline_recall)
print("Version 4 recall:", gb_metrics["recall"])

print("Promote Version 4:", version4_should_promote)

Baseline F1: 0.8866995073891626
Version 4 F1: 0.8115942028985508
Baseline recall: 0.9375
Version 4 recall: 0.875
Promote Version 4: False


# Version 5 – Feature-Selected Decision Tree

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectFromModel

feature_selected_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "feature_selection",
        SelectFromModel(
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
                n_jobs=1
            ),
            threshold="median"
        )
    ),
    (
        "classifier",
        DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42
        )
    )
])

feature_selected_pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('feature_selection',
                 SelectFromModel(estimator=RandomForestClassifier(class_weight='balanced',
                                                                  n_estimators=200,
                                                                  n_jobs=1,
                                                                  random_state=42),
                                 threshold='median')),
                ('classifier',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=8,
                                        min_samples_leaf=5, random_state=42))])

In [0]:
v5_predictions = feature_selected_pipeline.predict(X_test)
v5_probabilities = feature_selected_pipeline.predict_proba(X_test)[:, 1]

v5_metrics = {
    "accuracy": accuracy_score(y_test, v5_predictions),
    "precision": precision_score(
        y_test,
        v5_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        v5_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        v5_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        v5_probabilities
    )
}

print("Version 5 metrics:")
print(v5_metrics)

print("\nClassification report:")
print(classification_report(
    y_test,
    v5_predictions,
    digits=4
))

print("Confusion matrix:")
print(confusion_matrix(
    y_test,
    v5_predictions
))

Version 5 metrics:
{'accuracy': 0.7818181818181819, 'precision': 0.8260869565217391, 'recall': 0.7916666666666666, 'f1_score': 0.8085106382978723, 'roc_auc': np.float64(0.7876660628019324)}

Classification report:
              precision    recall  f1-score   support

           0     0.7260    0.7681    0.7465        69
           1     0.8261    0.7917    0.8085        96

    accuracy                         0.7818       165
   macro avg     0.7761    0.7799    0.7775       165
weighted avg     0.7842    0.7818    0.7826       165

Confusion matrix:
[[53 16]
 [20 76]]


In [0]:
version5_should_promote = (
    v5_metrics["f1_score"] > baseline_f1
    and v5_metrics["recall"] >= baseline_recall - 0.02
)

print("Baseline F1:", baseline_f1)
print("Version 5 F1:", v5_metrics["f1_score"])

print("Baseline recall:", baseline_recall)
print("Version 5 recall:", v5_metrics["recall"])

print("Promote Version 5:", version5_should_promote)

Baseline F1: 0.8866995073891626
Version 5 F1: 0.8085106382978723
Baseline recall: 0.9375
Version 5 recall: 0.7916666666666666
Promote Version 5: False


# Version 6 – Extra Trees Ensemble

In [0]:
from sklearn.model_selection import GroupShuffleSplit

validation_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=123
)

subtrain_idx, validation_idx = next(
    validation_splitter.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

X_subtrain = X_train.iloc[subtrain_idx].copy()
X_validation = X_train.iloc[validation_idx].copy()

y_subtrain = y_train.iloc[subtrain_idx].copy()
y_validation = y_train.iloc[validation_idx].copy()

groups_subtrain = groups_train.iloc[subtrain_idx]
groups_validation = groups_train.iloc[validation_idx]

print("Subtrain:", X_subtrain.shape)
print("Validation:", X_validation.shape)

print(
    "Group overlap:",
    set(groups_subtrain).intersection(
        set(groups_validation)
    )
)

print("\nValidation target distribution:")
print(y_validation.value_counts())

Subtrain: (741, 190)
Validation: (201, 190)
Group overlap: set()

Validation target distribution:
target
0    127
1     74
Name: count, dtype: int64


In [0]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier

extra_tree_candidates = {
    "extra_trees_a": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),

    "extra_trees_b": ExtraTreesClassifier(
        n_estimators=500,
        max_depth=16,
        min_samples_leaf=3,
        max_features=0.5,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ),

    "extra_trees_c": ExtraTreesClassifier(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=2,
        max_features="log2",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=1
    )
}

In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

validation_results = []
candidate_pipelines = {}

for model_name, classifier in extra_tree_candidates.items():

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", classifier)
    ])

    pipeline.fit(X_subtrain, y_subtrain)

    probabilities = pipeline.predict_proba(
        X_validation
    )[:, 1]

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(
            y_validation,
            predictions
        ),
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            probabilities
        )
    }

    validation_results.append(metrics)
    candidate_pipelines[model_name] = pipeline

validation_results_df = pd.DataFrame(validation_results)

display(
    validation_results_df.sort_values(
        by=["f1_score", "recall"],
        ascending=False
    )
)

model,accuracy,precision,recall,f1_score,roc_auc
extra_trees_b,0.7711442786069652,0.868421052631579,0.44594594594594594,0.5892857142857143,0.9597786763141094
extra_trees_a,0.7761194029850746,0.9142857142857143,0.43243243243243246,0.5871559633027523,0.9538199616939775
extra_trees_c,0.7711442786069652,0.8888888888888888,0.43243243243243246,0.5818181818181818,0.9548840178761439


In [0]:
best_candidate_name = (
    validation_results_df
    .sort_values(
        by=["f1_score", "recall"],
        ascending=False
    )
    .iloc[0]["model"]
)

best_extra_pipeline = candidate_pipelines[
    best_candidate_name
]

validation_probabilities = (
    best_extra_pipeline.predict_proba(
        X_validation
    )[:, 1]
)

threshold_results = []

for threshold in np.arange(0.25, 0.76, 0.025):

    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": float(threshold),
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_validation,
            predictions,
            zero_division=0
        )
    })

threshold_results_df = pd.DataFrame(
    threshold_results
)

eligible_thresholds = threshold_results_df[
    threshold_results_df["recall"] >= 0.90
]

if len(eligible_thresholds) > 0:
    selected_threshold_row = (
        eligible_thresholds
        .sort_values(
            by=["f1_score", "recall"],
            ascending=False
        )
        .iloc[0]
    )
else:
    selected_threshold_row = (
        threshold_results_df
        .sort_values(
            by=["f1_score", "recall"],
            ascending=False
        )
        .iloc[0]
    )

selected_threshold = float(
    selected_threshold_row["threshold"]
)

print("Best candidate:", best_candidate_name)
print("Selected threshold:", selected_threshold)

display(
    threshold_results_df.sort_values(
        by="f1_score",
        ascending=False
    ).head(10)
)

Best candidate: extra_trees_b
Selected threshold: 0.25


threshold,precision,recall,f1_score
0.25,0.8813559322033898,0.7027027027027027,0.7819548872180451
0.275,0.8771929824561403,0.6756756756756757,0.7633587786259542
0.30000000000000004,0.8703703703703703,0.6351351351351351,0.734375
0.32500000000000007,0.8653846153846154,0.6081081081081081,0.7142857142857143
0.3500000000000001,0.8627450980392157,0.5945945945945946,0.704
0.3750000000000001,0.875,0.5675675675675675,0.6885245901639344
0.40000000000000013,0.8695652173913043,0.5405405405405406,0.6666666666666666
0.42500000000000016,0.8863636363636364,0.527027027027027,0.6610169491525424
0.4500000000000002,0.8809523809523809,0.5,0.6379310344827587
0.4750000000000002,0.8717948717948718,0.4594594594594595,0.6017699115044248


In [0]:
best_classifier = extra_tree_candidates[
    best_candidate_name
]

version6_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", best_classifier)
])

version6_model.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('classifier',
                 ExtraTreesClassifier(class_weight='balanced', max_depth=16,
                                      max_features=0.5, min_samples_leaf=3,
                                      n_estimators=500, n_jobs=1,
                                      random_state=42))])

In [0]:
version6_probabilities = version6_model.predict_proba(
    X_test
)[:, 1]

version6_predictions = (
    version6_probabilities >= selected_threshold
).astype(int)

version6_metrics = {
    "accuracy": accuracy_score(
        y_test,
        version6_predictions
    ),
    "precision": precision_score(
        y_test,
        version6_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        version6_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        version6_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        version6_probabilities
    )
}

print("Version 6 metrics:")
print(version6_metrics)

print("\nClassification report:")
print(classification_report(
    y_test,
    version6_predictions,
    digits=4
))

print("Confusion matrix:")
print(confusion_matrix(
    y_test,
    version6_predictions
))

Version 6 metrics:
{'accuracy': 0.7333333333333333, 'precision': 0.6940298507462687, 'recall': 0.96875, 'f1_score': 0.808695652173913, 'roc_auc': np.float64(0.922403381642512)}

Classification report:
              precision    recall  f1-score   support

           0     0.9032    0.4058    0.5600        69
           1     0.6940    0.9688    0.8087        96

    accuracy                         0.7333       165
   macro avg     0.7986    0.6873    0.6843       165
weighted avg     0.7815    0.7333    0.7047       165

Confusion matrix:
[[28 41]
 [ 3 93]]


In [0]:
version6_should_promote = (
    version6_metrics["f1_score"] > baseline_f1
    and
    version6_metrics["recall"] >= baseline_recall - 0.02
)

print("Baseline F1:", baseline_f1)
print("Version 6 F1:", version6_metrics["f1_score"])

print("Baseline recall:", baseline_recall)
print("Version 6 recall:", version6_metrics["recall"])

print("Promote Version 6:", version6_should_promote)

Baseline F1: 0.8866995073891626
Version 6 F1: 0.808695652173913
Baseline recall: 0.9375
Version 6 recall: 0.96875
Promote Version 6: False


# Version 7 – XGBoost with Threshold Optimization

In [0]:
try:
    import xgboost
    print("XGBoost version:", xgboost.__version__)
except ImportError:
    print("XGBoost is not installed.")

XGBoost is not installed.


In [0]:
%pip install -q xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
xgb_candidates = [
    {
        "name": "xgb_a",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 1
    },
    {
        "name": "xgb_b",
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 3,
        "subsample": 0.85,
        "colsample_bytree": 0.75,
        "gamma": 0.1,
        "reg_alpha": 0.05,
        "reg_lambda": 3
    },
    {
        "name": "xgb_c",
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "gamma": 0.1,
        "reg_alpha": 0.1,
        "reg_lambda": 5
    }
]

In [0]:
from sklearn.model_selection import GroupShuffleSplit

validation_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=123
)

subtrain_idx, validation_idx = next(
    validation_splitter.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

X_subtrain = X_train.iloc[subtrain_idx].copy()
X_validation = X_train.iloc[validation_idx].copy()

y_subtrain = y_train.iloc[subtrain_idx].copy()
y_validation = y_train.iloc[validation_idx].copy()

groups_subtrain = groups_train.iloc[subtrain_idx]
groups_validation = groups_train.iloc[validation_idx]

print("Subtrain shape:", X_subtrain.shape)
print("Validation shape:", X_validation.shape)

print(
    "Experiment overlap:",
    set(groups_subtrain).intersection(set(groups_validation))
)

print("\nSubtrain target:")
print(y_subtrain.value_counts())

print("\nValidation target:")
print(y_validation.value_counts())

Subtrain shape: (734, 190)
Validation shape: (208, 190)
Experiment overlap: set()

Subtrain target:
target
0    612
1    122
Name: count, dtype: int64

Validation target:
target
0    133
1     75
Name: count, dtype: int64


In [0]:
from sklearn.impute import SimpleImputer
import pandas as pd

validation_imputer = SimpleImputer(strategy="median")

X_subtrain_imputed = pd.DataFrame(
    validation_imputer.fit_transform(X_subtrain),
    columns=X_subtrain.columns,
    index=X_subtrain.index
)

X_validation_imputed = pd.DataFrame(
    validation_imputer.transform(X_validation),
    columns=X_validation.columns,
    index=X_validation.index
)

In [0]:
negative_count = (y_subtrain == 0).sum()
positive_count = (y_subtrain == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative samples:", negative_count)
print("Positive samples:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Negative samples: 612
Positive samples: 122
scale_pos_weight: 5.016393442622951


In [0]:
from xgboost import XGBClassifier

xgb_candidates = [
    {
        "name": "xgb_a",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 1
    },
    {
        "name": "xgb_b",
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 3,
        "subsample": 0.85,
        "colsample_bytree": 0.75,
        "gamma": 0.1,
        "reg_alpha": 0.05,
        "reg_lambda": 3
    },
    {
        "name": "xgb_c",
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "gamma": 0.1,
        "reg_alpha": 0.1,
        "reg_lambda": 5
    },
    {
        "name": "xgb_d",
        "n_estimators": 400,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 1.0,
        "colsample_bytree": 0.7,
        "gamma": 0.2,
        "reg_alpha": 0.1,
        "reg_lambda": 5
    },
    {
        "name": "xgb_e",
        "n_estimators": 200,
        "max_depth": 3,
        "learning_rate": 0.08,
        "min_child_weight": 1,
        "subsample": 0.9,
        "colsample_bytree": 1.0,
        "gamma": 0,
        "reg_alpha": 0,
        "reg_lambda": 3
    }
]

In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

candidate_models = {}
candidate_results = []

for candidate in xgb_candidates:

    candidate_name = candidate["name"]

    model_parameters = {
        key: value
        for key, value in candidate.items()
        if key != "name"
    }

    model = XGBClassifier(
        **model_parameters,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        n_jobs=1,
        random_state=42,
        verbosity=0
    )

    model.fit(
        X_subtrain_imputed,
        y_subtrain
    )

    validation_probabilities = model.predict_proba(
        X_validation_imputed
    )[:, 1]

    validation_predictions = (
        validation_probabilities >= 0.50
    ).astype(int)

    candidate_results.append({
        "model": candidate_name,
        "accuracy": accuracy_score(
            y_validation,
            validation_predictions
        ),
        "precision": precision_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            validation_probabilities
        )
    })

    candidate_models[candidate_name] = model

xgb_validation_results = pd.DataFrame(candidate_results)

display(
    xgb_validation_results.sort_values(
        by=["f1_score", "recall", "roc_auc"],
        ascending=False
    )
)

model,accuracy,precision,recall,f1_score,roc_auc
xgb_d,0.8461538461538461,0.9387755102040817,0.6133333333333333,0.7419354838709677,0.9676190476190476
xgb_c,0.8413461538461539,0.92,0.6133333333333333,0.736,0.9658145363408521
xgb_b,0.8413461538461539,0.9375,0.6,0.7317073170731707,0.9656140350877194
xgb_a,0.8365384615384616,0.9361702127659575,0.5866666666666667,0.7213114754098361,0.9712280701754387
xgb_e,0.8317307692307693,0.9347826086956522,0.5733333333333334,0.7107438016528925,0.967218045112782


In [0]:
eligible_candidates = xgb_validation_results[
    xgb_validation_results["recall"] >= 0.85
]

if not eligible_candidates.empty:
    best_xgb_row = eligible_candidates.sort_values(
        by=["f1_score", "recall", "roc_auc"],
        ascending=False
    ).iloc[0]
else:
    best_xgb_row = xgb_validation_results.sort_values(
        by=["f1_score", "recall", "roc_auc"],
        ascending=False
    ).iloc[0]

best_xgb_name = best_xgb_row["model"]
best_validation_model = candidate_models[best_xgb_name]

print("Selected candidate:", best_xgb_name)
print(best_xgb_row)

Selected candidate: xgb_d
model           xgb_d
accuracy     0.846154
precision    0.938776
recall       0.613333
f1_score     0.741935
roc_auc      0.967619
Name: 3, dtype: object


In [0]:
import numpy as np

validation_probabilities = best_validation_model.predict_proba(
    X_validation_imputed
)[:, 1]

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.02):

    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": float(threshold),
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_validation,
            predictions,
            zero_division=0
        )
    })

xgb_threshold_results = pd.DataFrame(threshold_results)

eligible_thresholds = xgb_threshold_results[
    xgb_threshold_results["recall"] >= 0.90
]

if not eligible_thresholds.empty:
    selected_threshold_row = eligible_thresholds.sort_values(
        by=["f1_score", "precision", "recall"],
        ascending=False
    ).iloc[0]
else:
    selected_threshold_row = xgb_threshold_results.sort_values(
        by=["f1_score", "recall"],
        ascending=False
    ).iloc[0]

selected_xgb_threshold = float(
    selected_threshold_row["threshold"]
)

print("Selected threshold:", selected_xgb_threshold)
print(selected_threshold_row)

display(
    xgb_threshold_results.sort_values(
        by="f1_score",
        ascending=False
    ).head(10)
)

Selected threshold: 0.2
threshold    0.200000
precision    0.924528
recall       0.653333
f1_score     0.765625
Name: 0, dtype: float64


threshold,precision,recall,f1_score
0.2,0.9245283018867925,0.6533333333333333,0.765625
0.22,0.9245283018867925,0.6533333333333333,0.765625
0.24,0.9230769230769231,0.64,0.7559055118110236
0.26,0.9230769230769231,0.64,0.7559055118110236
0.27999999999999997,0.9230769230769231,0.64,0.7559055118110236
0.29999999999999993,0.9230769230769231,0.64,0.7559055118110236
0.31999999999999995,0.9230769230769231,0.64,0.7559055118110236
0.33999999999999997,0.9215686274509803,0.6266666666666667,0.746031746031746
0.35999999999999993,0.9215686274509803,0.6266666666666667,0.746031746031746
0.3799999999999999,0.9215686274509803,0.6266666666666667,0.746031746031746


In [0]:
final_imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    final_imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    final_imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

training_negative_count = (y_train == 0).sum()
training_positive_count = (y_train == 1).sum()

final_scale_pos_weight = (
    training_negative_count / training_positive_count
)

selected_candidate_parameters = next(
    {
        key: value
        for key, value in candidate.items()
        if key != "name"
    }
    for candidate in xgb_candidates
    if candidate["name"] == best_xgb_name
)

version7_xgb = XGBClassifier(
    **selected_candidate_parameters,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=final_scale_pos_weight,
    tree_method="hist",
    n_jobs=1,
    random_state=42,
    verbosity=0
)

version7_xgb.fit(
    X_train_imputed,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.2,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=2, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=1,
              num_parallel_tree=None, ...)

In [0]:
from sklearn.metrics import classification_report, confusion_matrix

version7_probabilities = version7_xgb.predict_proba(
    X_test_imputed
)[:, 1]

version7_predictions = (
    version7_probabilities >= selected_xgb_threshold
).astype(int)

version7_metrics = {
    "accuracy": accuracy_score(
        y_test,
        version7_predictions
    ),
    "precision": precision_score(
        y_test,
        version7_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        version7_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        version7_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        version7_probabilities
    )
}

print("Version 7 metrics:")
print(version7_metrics)

print("\nClassification report:")
print(classification_report(
    y_test,
    version7_predictions,
    digits=4
))

print("Confusion matrix:")
print(confusion_matrix(
    y_test,
    version7_predictions
))

Version 7 metrics:
{'accuracy': 0.7454545454545455, 'precision': 0.6985294117647058, 'recall': 0.9895833333333334, 'f1_score': 0.8189655172413793, 'roc_auc': np.float64(0.8964371980676329)}

Classification report:
              precision    recall  f1-score   support

           0     0.9655    0.4058    0.5714        69
           1     0.6985    0.9896    0.8190        96

    accuracy                         0.7455       165
   macro avg     0.8320    0.6977    0.6952       165
weighted avg     0.8102    0.7455    0.7155       165

Confusion matrix:
[[28 41]
 [ 1 95]]


In [0]:
version7_should_promote = (
    version7_metrics["f1_score"] > baseline_f1
    and
    version7_metrics["recall"] >= baseline_recall - 0.02
)

print("Baseline F1:", baseline_f1)
print("Version 7 F1:", version7_metrics["f1_score"])

print("Baseline recall:", baseline_recall)
print("Version 7 recall:", version7_metrics["recall"])

print("Promote Version 7:", version7_should_promote)

Baseline F1: 0.8866995073891626
Version 7 F1: 0.8189655172413793
Baseline recall: 0.9375
Version 7 recall: 0.9895833333333334
Promote Version 7: False


In [0]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

version8_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "classifier",
        DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42
        )
    )
])

In [0]:
from sklearn.model_selection import GroupKFold, cross_val_predict

group_cv = GroupKFold(n_splits=5)

oof_probabilities = cross_val_predict(
    version8_pipeline,
    X_train,
    y_train,
    groups=groups_train,
    cv=group_cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("OOF predictions created:", len(oof_probabilities))

OOF predictions created: 942


In [0]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

threshold_results = []

for threshold in np.arange(0.20, 0.91, 0.01):

    predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": float(threshold),
        "precision": precision_score(
            y_train,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_train,
            predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_train,
            predictions,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

eligible_thresholds = threshold_df[
    threshold_df["recall"] >= 0.92
]

if not eligible_thresholds.empty:
    selected_row = eligible_thresholds.sort_values(
        by=["f1_score", "precision", "recall"],
        ascending=False
    ).iloc[0]
else:
    selected_row = threshold_df.sort_values(
        by=["f1_score", "recall"],
        ascending=False
    ).iloc[0]

version8_threshold = float(selected_row["threshold"])

print("Selected threshold:", version8_threshold)
print(selected_row)

display(
    threshold_df.sort_values(
        by="f1_score",
        ascending=False
    ).head(10)
)

Selected threshold: 0.45000000000000023
threshold    0.450000
precision    0.695187
recall       0.659898
f1_score     0.677083
Name: 25, dtype: float64


threshold,precision,recall,f1_score
0.46000000000000024,0.6951871657754011,0.6598984771573604,0.6770833333333334
0.47000000000000025,0.6951871657754011,0.6598984771573604,0.6770833333333334
0.45000000000000023,0.6951871657754011,0.6598984771573604,0.6770833333333334
0.23000000000000004,0.6735751295336787,0.6598984771573604,0.6666666666666666
0.2,0.6735751295336787,0.6598984771573604,0.6666666666666666
0.21000000000000002,0.6735751295336787,0.6598984771573604,0.6666666666666666
0.22000000000000003,0.6735751295336787,0.6598984771573604,0.6666666666666666
0.26000000000000006,0.6735751295336787,0.6598984771573604,0.6666666666666666
0.25000000000000006,0.6735751295336787,0.6598984771573604,0.6666666666666666
0.24000000000000005,0.6735751295336787,0.6598984771573604,0.6666666666666666


In [0]:
version8_pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('classifier',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=8,
                                        min_samples_leaf=5, random_state=42))])

In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

version8_probabilities = version8_pipeline.predict_proba(
    X_test
)[:, 1]

version8_predictions = (
    version8_probabilities >= version8_threshold
).astype(int)

version8_metrics = {
    "accuracy": accuracy_score(
        y_test,
        version8_predictions
    ),
    "precision": precision_score(
        y_test,
        version8_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        version8_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        version8_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        version8_probabilities
    )
}

print("Version 8 threshold:", version8_threshold)
print("Version 8 metrics:", version8_metrics)

print("\nClassification report:")
print(classification_report(
    y_test,
    version8_predictions,
    digits=4
))

print("Confusion matrix:")
print(confusion_matrix(
    y_test,
    version8_predictions
))

Version 8 threshold: 0.45000000000000023
Version 8 metrics: {'accuracy': 0.8545454545454545, 'precision': 0.8333333333333334, 'recall': 0.9375, 'f1_score': 0.8823529411764706, 'roc_auc': np.float64(0.8463164251207729)}

Classification report:
              precision    recall  f1-score   support

           0     0.8947    0.7391    0.8095        69
           1     0.8333    0.9375    0.8824        96

    accuracy                         0.8545       165
   macro avg     0.8640    0.8383    0.8459       165
weighted avg     0.8590    0.8545    0.8519       165

Confusion matrix:
[[51 18]
 [ 6 90]]


In [0]:
version8_should_promote = (
    version8_metrics["f1_score"] > baseline_f1
    and
    version8_metrics["recall"] >= baseline_recall - 0.02
)

print("Baseline F1:", baseline_f1)
print("Version 8 F1:", version8_metrics["f1_score"])

print("Baseline recall:", baseline_recall)
print("Version 8 recall:", version8_metrics["recall"])

print("Promote Version 8:", version8_should_promote)

Baseline F1: 0.8866995073891626
Version 8 F1: 0.8823529411764706
Baseline recall: 0.9375
Version 8 recall: 0.9375
Promote Version 8: False


In [0]:
from sklearn.model_selection import GroupShuffleSplit

# 20% untouched final holdout
outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=2026
)

development_idx, final_idx = next(
    outer_split.split(X, y, groups=groups)
)

X_development = X.iloc[development_idx].copy()
X_final = X.iloc[final_idx].copy()

y_development = y.iloc[development_idx].copy()
y_final = y.iloc[final_idx].copy()

groups_development = groups.iloc[development_idx]
groups_final = groups.iloc[final_idx]

# Split development data into train and validation
inner_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=2027
)

train_idx, validation_idx = next(
    inner_split.split(
        X_development,
        y_development,
        groups=groups_development
    )
)

X_train_v9 = X_development.iloc[train_idx].copy()
X_validation_v9 = X_development.iloc[validation_idx].copy()

y_train_v9 = y_development.iloc[train_idx].copy()
y_validation_v9 = y_development.iloc[validation_idx].copy()

groups_train_v9 = groups_development.iloc[train_idx]
groups_validation_v9 = groups_development.iloc[validation_idx]

print("Train:", X_train_v9.shape)
print("Validation:", X_validation_v9.shape)
print("Final holdout:", X_final.shape)

print(
    "Train-validation overlap:",
    set(groups_train_v9).intersection(groups_validation_v9)
)

print(
    "Development-final overlap:",
    set(groups_development).intersection(groups_final)
)

Train: (587, 190)
Validation: (263, 190)
Final holdout: (257, 190)
Train-validation overlap: set()
Development-final overlap: set()


In [0]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier

decision_tree_v9 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ))
])

extra_trees_v9 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", ExtraTreesClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ))
])

negative_count = (y_train_v9 == 0).sum()
positive_count = (y_train_v9 == 1).sum()

xgboost_v9 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", XGBClassifier(
        n_estimators=250,
        max_depth=2,
        learning_rate=0.05,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=negative_count / positive_count,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1,
        verbosity=0
    ))
])

In [0]:
decision_tree_v9.fit(X_train_v9, y_train_v9)
extra_trees_v9.fit(X_train_v9, y_train_v9)
xgboost_v9.fit(X_train_v9, y_train_v9)

dt_val_probability = decision_tree_v9.predict_proba(
    X_validation_v9
)[:, 1]

et_val_probability = extra_trees_v9.predict_proba(
    X_validation_v9
)[:, 1]

xgb_val_probability = xgboost_v9.predict_proba(
    X_validation_v9
)[:, 1]

In [0]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

ensemble_search_results = []

for dt_weight in [0.4, 0.5, 0.6, 0.7, 0.8]:
    for et_weight in [0.0, 0.1, 0.2, 0.3]:

        xgb_weight = 1.0 - dt_weight - et_weight

        if xgb_weight < 0:
            continue

        ensemble_probability = (
            dt_weight * dt_val_probability
            + et_weight * et_val_probability
            + xgb_weight * xgb_val_probability
        )

        for threshold in np.arange(0.30, 0.71, 0.025):

            predictions = (
                ensemble_probability >= threshold
            ).astype(int)

            ensemble_search_results.append({
                "dt_weight": dt_weight,
                "et_weight": et_weight,
                "xgb_weight": xgb_weight,
                "threshold": float(threshold),
                "precision": precision_score(
                    y_validation_v9,
                    predictions,
                    zero_division=0
                ),
                "recall": recall_score(
                    y_validation_v9,
                    predictions,
                    zero_division=0
                ),
                "f1_score": f1_score(
                    y_validation_v9,
                    predictions,
                    zero_division=0
                )
            })

ensemble_search_df = pd.DataFrame(
    ensemble_search_results
)

eligible_results = ensemble_search_df[
    ensemble_search_df["recall"] >= 0.90
]

if not eligible_results.empty:
    best_ensemble_settings = (
        eligible_results
        .sort_values(
            by=["f1_score", "precision", "recall"],
            ascending=False
        )
        .iloc[0]
    )
else:
    best_ensemble_settings = (
        ensemble_search_df
        .sort_values(
            by=["f1_score", "recall"],
            ascending=False
        )
        .iloc[0]
    )

print("Selected ensemble settings:")
print(best_ensemble_settings)

Selected ensemble settings:
dt_weight     0.400000
et_weight     0.000000
xgb_weight    0.600000
threshold     0.400000
precision     0.825000
recall        0.933962
f1_score      0.876106
Name: 4, dtype: float64


In [0]:
X_development_v9 = pd.concat(
    [X_train_v9, X_validation_v9]
)

y_development_v9 = pd.concat(
    [y_train_v9, y_validation_v9]
)

development_negative = (y_development_v9 == 0).sum()
development_positive = (y_development_v9 == 1).sum()

decision_tree_v9.fit(
    X_development_v9,
    y_development_v9
)

extra_trees_v9.fit(
    X_development_v9,
    y_development_v9
)

xgboost_v9.set_params(
    classifier__scale_pos_weight=(
        development_negative / development_positive
    )
)

xgboost_v9.fit(
    X_development_v9,
    y_development_v9
)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=2, max_leaves=None, min_child_weight=1,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=250, n_jobs=1,
                               num_parallel_tree=None, ...))])

In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

dt_final_probability = decision_tree_v9.predict_proba(
    X_final
)[:, 1]

et_final_probability = extra_trees_v9.predict_proba(
    X_final
)[:, 1]

xgb_final_probability = xgboost_v9.predict_proba(
    X_final
)[:, 1]

version9_probability = (
    best_ensemble_settings["dt_weight"]
    * dt_final_probability
    + best_ensemble_settings["et_weight"]
    * et_final_probability
    + best_ensemble_settings["xgb_weight"]
    * xgb_final_probability
)

version9_predictions = (
    version9_probability
    >= best_ensemble_settings["threshold"]
).astype(int)

version9_metrics = {
    "accuracy": accuracy_score(
        y_final,
        version9_predictions
    ),
    "precision": precision_score(
        y_final,
        version9_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_final,
        version9_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_final,
        version9_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_final,
        version9_probability
    )
}

print("Version 9 metrics:")
print(version9_metrics)

print("\nClassification report:")
print(classification_report(
    y_final,
    version9_predictions,
    digits=4
))

print("Confusion matrix:")
print(confusion_matrix(
    y_final,
    version9_predictions
))

Version 9 metrics:
{'accuracy': 0.8754863813229572, 'precision': 0.7241379310344828, 'recall': 1.0, 'f1_score': 0.84, 'roc_auc': np.float64(0.9820396366639141)}

Classification report:
              precision    recall  f1-score   support

           0     1.0000    0.8150    0.8981       173
           1     0.7241    1.0000    0.8400        84

    accuracy                         0.8755       257
   macro avg     0.8621    0.9075    0.8690       257
weighted avg     0.9098    0.8755    0.8791       257

Confusion matrix:
[[141  32]
 [  0  84]]


In [0]:
baseline_v9 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ))
])

baseline_v9.fit(
    X_development_v9,
    y_development_v9
)

baseline_v9_predictions = baseline_v9.predict(
    X_final
)

baseline_v9_probabilities = baseline_v9.predict_proba(
    X_final
)[:, 1]

baseline_v9_metrics = {
    "accuracy": accuracy_score(
        y_final,
        baseline_v9_predictions
    ),
    "precision": precision_score(
        y_final,
        baseline_v9_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_final,
        baseline_v9_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_final,
        baseline_v9_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_final,
        baseline_v9_probabilities
    )
}

print("Fresh-holdout baseline:")
print(baseline_v9_metrics)

print("\nFresh-holdout Version 9:")
print(version9_metrics)

Fresh-holdout baseline:
{'accuracy': 0.9182879377431906, 'precision': 0.8247422680412371, 'recall': 0.9523809523809523, 'f1_score': 0.8839779005524862, 'roc_auc': np.float64(0.927401596476741)}

Fresh-holdout Version 9:
{'accuracy': 0.8754863813229572, 'precision': 0.7241379310344828, 'recall': 1.0, 'f1_score': 0.84, 'roc_auc': np.float64(0.9820396366639141)}


In [0]:
version9_should_promote = (
    version9_metrics["f1_score"]
    > baseline_v9_metrics["f1_score"]
    and
    version9_metrics["recall"]
    >= baseline_v9_metrics["recall"] - 0.02
)

print(
    "Baseline final F1:",
    baseline_v9_metrics["f1_score"]
)

print(
    "Version 9 final F1:",
    version9_metrics["f1_score"]
)

print(
    "Baseline final recall:",
    baseline_v9_metrics["recall"]
)

print(
    "Version 9 final recall:",
    version9_metrics["recall"]
)

print("Promote Version 9:", version9_should_promote)

Baseline final F1: 0.8839779005524862
Version 9 final F1: 0.84
Baseline final recall: 0.9523809523809523
Version 9 final recall: 1.0
Promote Version 9: False


In [0]:
from sklearn.metrics import confusion_matrix
import pandas as pd

baseline_cm = confusion_matrix(
    y_final,
    baseline_v9_predictions
)

version9_cm = confusion_matrix(
    y_final,
    version9_predictions
)

baseline_tn, baseline_fp, baseline_fn, baseline_tp = baseline_cm.ravel()
v9_tn, v9_fp, v9_fn, v9_tp = version9_cm.ravel()

comparison = pd.DataFrame([
    {
        "model": "Baseline Champion",
        "true_negative": baseline_tn,
        "false_positive": baseline_fp,
        "false_negative": baseline_fn,
        "true_positive": baseline_tp,
        "f1": baseline_v9_metrics["f1_score"],
        "recall": baseline_v9_metrics["recall"]
    },
    {
        "model": "Version 9 Ensemble",
        "true_negative": v9_tn,
        "false_positive": v9_fp,
        "false_negative": v9_fn,
        "true_positive": v9_tp,
        "f1": version9_metrics["f1_score"],
        "recall": version9_metrics["recall"]
    }
])

display(comparison)

model,true_negative,false_positive,false_negative,true_positive,f1,recall
Baseline Champion,156,17,4,80,0.8839779005524862,0.9523809523809523
Version 9 Ensemble,141,32,0,84,0.84,1.0


In [0]:
FALSE_NEGATIVE_COST = 10
FALSE_POSITIVE_COST = 1

baseline_operational_cost = (
    baseline_fn * FALSE_NEGATIVE_COST
    + baseline_fp * FALSE_POSITIVE_COST
)

version9_operational_cost = (
    v9_fn * FALSE_NEGATIVE_COST
    + v9_fp * FALSE_POSITIVE_COST
)

print("Baseline operational cost:", baseline_operational_cost)
print("Version 9 operational cost:", version9_operational_cost)

Baseline operational cost: 57
Version 9 operational cost: 32


In [0]:
version9_should_promote_cost_based = (
    version9_operational_cost < baseline_operational_cost
    and version9_metrics["recall"] > baseline_v9_metrics["recall"]
    and version9_metrics["precision"] >= 0.70
)

print(
    "Promote Version 9 using production-cost rule:",
    version9_should_promote_cost_based
)

Promote Version 9 using production-cost rule: True
